# Network Expansion
## Model 2 (Test) - Deterministic Multi-Year Extension

Sequential multi-year solve using the baseline MILP each year. Demand grows annually; first-stage decisions carry forward via updated network state (activated substations/lines become free, capacity reinforcements persist).

### 1 - Imports

In [4]:
import numpy as np
import pandas as pd

from src.classes import DistributionNetwork, Substation
from src.solver import solve_network, print_results

### 2 - Define the Distribution Network (shared across years)

In [5]:
# Nodes and loads
NODES = [f"N{i}" for i in range(1, 14)]
LOADS = [f"D{i}" for i in range(1, 11)]

# Initial substation
S1 = Substation("S1", "N4", 40, ['N3', 'N5', 'N9'], r_cost=200, edge_cost=50)
SUBSTATIONS = [S1]

line_cost = 50

base_load_capacity = {
    'D1': 6, 'D2': 3, 'D3': 2, 'D4': 5, 'D5': 3,
    'D6': 2, 'D7': 3, 'D8': 5, 'D9': 4, 'D10': 6
}

loads_locations = {
    'D1': 'N1', 'D2': 'N2', 'D3': 'N3', 'D4': 'N6', 'D5': 'N7',
    'D6': 'N8', 'D7': 'N9', 'D8': 'N11', 'D9': 'N12', 'D10': 'N13'
}

nodes_connected = {
    'N1': ['N2'],
    'N2': ['N1', 'N3'],
    'N3': ['N2', 'N4'],
    'N4': ['N3', 'N5', 'N9'],
    'N5': ['N4', 'N6'],
    'N6': ['N5','N7', 'N8'],
    'N7': ['N6'],
    'N8': ['N6'],
    'N9': ['N4', 'N10'],
    'N10': ['N9', 'N11', 'N13'],
    'N11': ['N10', 'N12'],
    'N12': ['N11'],
    'N13': ['N10']
}

DistributionNetwork = DistributionNetwork(
    NODES.copy(),
    LOADS.copy(),
    SUBSTATIONS.copy(),
    base_load_capacity.copy(),
    nodes_connected.copy(),
    loads_locations.copy(),
    line_cost
)

# Candidate substations (same as Model 1)
capacity = 15
s_cost = 100       # activation cost
l_cost = line_cost # feeder line cost
r_cost = 200       # capacity reinforcement cost

S2 = Substation("S2", "N14", capacity, ['N2'], r_cost, edge_cost=l_cost, fix_cost=s_cost)
S3 = Substation("S3", "N15", capacity, ['N6'], r_cost, edge_cost=l_cost, fix_cost=s_cost)
S4 = Substation("S4", "N16", capacity, ['N11', 'N13'], r_cost, edge_cost=l_cost, fix_cost=s_cost)

DistributionNetwork.add_candidate_substations([S2, S3, S4])

### 3 - Multi-year deterministic run

In [6]:
years = list(range(1, 11))      # Years 1..10
demand_rate = 0.06                 # annual growth
R = 10                             # size of capacity reinforcement
B = 550                            # annual budget

results = {}
system_demand = {}

# Year 1 uses base demand
system_demand[1] = sum(DistributionNetwork.load_capacity.values())
solution = solve_network(DistributionNetwork, R, B, OutputFlag=1)
results[1] = solution
print('Year 1')
print_results(DistributionNetwork, solution, detailed=True)
DistributionNetwork.update_initial_conditions(solution['w'], solution['x'], solution['z'], R)

# Years 2..T
for y in years[1:]:
    # grow demand
    for load, val in DistributionNetwork.load_capacity.items():
        DistributionNetwork.load_capacity[load] = (1 + demand_rate) * val
    system_demand[y] = sum(DistributionNetwork.load_capacity.values())

    sol = solve_network(DistributionNetwork, R, B, OutputFlag=0)
    results[y] = sol
    print(f'Year {y}')
    print_results(DistributionNetwork, sol, detailed=False)
    DistributionNetwork.update_initial_conditions(sol['w'], sol['x'], sol['z'], R)


Set parameter Username
Set parameter LicenseID to value 2706891
Academic license - for non-commercial use only - expires 2026-09-10
Set parameter OutputFlag to value 1
Gurobi Optimizer version 12.0.3 build v12.0.3rc0 (win64 - Windows 11.0 (26100.2))

CPU model: AMD Ryzen 5 4600H with Radeon Graphics, instruction set [SSE2|AVX|AVX2]
Thread count: 6 physical cores, 12 logical processors, using up to 12 threads

Optimize a model with 346 rows, 332 columns and 1309 nonzeros
Model fingerprint: 0x28f6dd4e
Model has 4 quadratic constraints
Variable types: 132 continuous, 200 integer (196 binary)
Coefficient statistics:
  Matrix range     [1e+00, 2e+02]
  QMatrix range    [1e+01, 1e+01]
  QLMatrix range   [1e+00, 4e+01]
  Objective range  [5e+01, 2e+02]
  Bounds range     [1e+00, 4e+01]
  RHS range        [1e+00, 6e+02]
Presolve removed 287 rows and 282 columns
Presolve time: 0.01s
Presolved: 59 rows, 50 columns, 200 nonzeros
Variable types: 12 continuous, 38 integer (33 binary)
Found heuristi

### 4 - Summary

In [7]:
years_idx = list(results.keys())
summary = pd.DataFrame({
    'System Demand': [round(system_demand[y], 2) for y in years_idx],
    'System Supply': [round(sum(results[y]['r'].values()), 2) for y in years_idx],
    'System Capacity': [sum(results[y]['P'].values()) for y in years_idx],
    'Substations Active': [[s for s, w in results[y]['w'].items() if w == 1] for y in years_idx],
    'Cost': [round(results[y]['objective'], 2) for y in years_idx]
}, index=years_idx)

summary

,System Demand,System Supply,System Capacity,Substations Active,Cost
1,39.00,39.00,40.0,[1],0.0
2,41.34,41.34,55.0,"[1, 4]",150.0
3,43.82,43.82,55.0,"[1, 4]",0.0
4,46.45,46.45,55.0,"[1, 4]",0.0
5,49.24,49.24,55.0,"[1, 4]",100.0
6,52.19,52.19,70.0,"[1, 2, 4]",150.0
7,55.32,55.32,70.0,"[1, 2, 4]",50.0
8,58.64,58.64,70.0,"[1, 2, 4]",0.0
9,62.16,62.16,70.0,"[1, 2, 4]",0.0
10,65.89,65.89,80.0,"[1, 2, 4]",350.0


### 5 - Final year assignment (Year 10)

In [8]:
last_year = max(results.keys())
S = list(np.arange(1, len(DistributionNetwork.SUBSTATIONS)+1))
N = list(np.arange(1, len(DistributionNetwork.NODES)+1))
print(f"Node assignments in Year {last_year}:")
for n in N:
    for s in S:
        if results[last_year]['y'][(n, s)] > 0.5:
            print(f"Node {DistributionNetwork.NODES[n-1]} assigned to {DistributionNetwork.SUBSTATIONS[s-1].id}")


Node assignments in Year 10:
Node N1 assigned to S2
Node N2 assigned to S2
Node N3 assigned to S2
Node N4 assigned to S1
Node N5 assigned to S1
Node N6 assigned to S1
Node N7 assigned to S1
Node N8 assigned to S1
Node N9 assigned to S1
Node N10 assigned to S1
Node N11 assigned to S1
Node N12 assigned to S1
Node N13 assigned to S4
Node N14 assigned to S2
Node N16 assigned to S4
